# Milestone 2 — Hugging Face Transformers, Attention & Zero-Shot Classification


In [22]:
!pip install datasets transformers sentence-transformers --quiet

## Q1 — Load data with Hugging Face `datasets` (not pandas) + `.map()`

**What's happening:** Instead of `pandas.read_csv`, we use Hugging Face's `datasets` library, which is built to handle large datasets efficiently and plugs directly into the rest of the Hugging Face ecosystem. `.map()` lets us apply a function to every row at once — here, we're building a new `combined_text` column by joining the `prompt` and `A` columns with a space.

In [2]:
from datasets import load_dataset

# Load the CSV as a Hugging Face Dataset object (not a pandas DataFrame)
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

# .map() applies this function to every single row
def add_combined_text(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(add_combined_text)

# Zero-indexing: row 51 means the 52nd row
combined_text_51 = dataset[51]["combined_text"]
print(combined_text_51)
print("Character length:", len(combined_text_51))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length: 614


## Q2 — BERT tokenizer vocabulary size

**What's happening:** A tokenizer's job is to break text into smaller pieces ("tokens") that the model understands, and convert each token into a number (an ID). Every tokenizer has a fixed **vocabulary** — a hardcoded list of all the tokens it knows. `bert-base-uncased` is a specific pretrained BERT tokenizer that lowercases everything ("uncased").

In [3]:
from transformers import AutoTokenizer

# Download and load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# vocab_size tells us how many unique tokens this tokenizer can represent
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 30522


## Q3 — The [SEP] special token's ID

**What's happening:** BERT uses special tokens to mark structure in text. `[SEP]` ("separator") tells the model where one sentence ends — for example, when you feed it two sentences to compare. Every special token has a fixed integer ID in the vocabulary, just like regular words do.

In [4]:
sep_token_id = tokenizer.sep_token_id
print("[SEP] token ID:", sep_token_id)

[SEP] token ID: 102


## Q4 — Tokenizing the entire prompt column at once

**What's happening:** We tokenize *every* prompt in the dataset in a single call, with three important settings:
- `padding="max_length"` → shorter prompts get padded with zeros so every sequence is the same length
- `truncation=True` → longer prompts get cut off
- `max_length=128` → the fixed length every sequence gets padded/truncated to
- `return_tensors="pt"` → return PyTorch tensors instead of plain Python lists

The result, `input_ids`, is one row of token IDs per prompt — so its shape should be `(number_of_prompts, 128)`.

In [24]:
prompts = list(dataset["prompt"])  # force a plain Python list, not a Dataset column wrapper

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
print("input_ids shape:", encoded["input_ids"].shape)

input_ids shape: torch.Size([2000, 128])


## Q5 — Dimensionality of a single attention head

**What's happening:** BERT doesn't use one big "attention" calculation — it splits its 768-dimensional hidden size into 12 smaller, parallel "attention heads," each focusing on different patterns in the text. Since the hidden size is split *equally*, each head's dimensionality is simply `hidden_size ÷ number_of_heads`.

In [8]:
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads
print("Dimensionality of each attention head:", head_dim)

Dimensionality of each attention head: 64


## Q6 — Shape of `last_hidden_state`

**What's happening:** Now we load the actual BERT *model* (not just the tokenizer) and run a real prompt through it. `last_hidden_state` is the model's final output — one 768-dimensional vector *per token* in the input. Since we're not manually padding/truncating this time, the sequence length will just be whatever this specific prompt naturally tokenizes to (plus the `[CLS]` and `[SEP]` special tokens BERT always adds).

In [9]:
from transformers import AutoModel
import torch

# Load the pretrained BERT model itself
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()  # inference mode — turns off things like dropout used only during training

row0_prompt = dataset[0]["prompt"]

# Default settings: no manual padding or truncation
inputs = tokenizer(row0_prompt, return_tensors="pt")

# torch.no_grad() = don't track gradients, since we're not training anything
with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
print("last_hidden_state shape:", last_hidden_state.shape)
# Shape is (batch_size, sequence_length, hidden_size) -> (1, num_tokens, 768)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])


## Q7 — Sum of the first 5 values in the [CLS] embedding

**What's happening:** The very first token BERT adds to any input is `[CLS]` ("classification"), always at position/index 0. Its final embedding is often used as a summary representation of the *entire* input sentence — that's why it's commonly used for classification tasks.

In [10]:
# [0, 0, :] means: first item in the batch, token at position 0 (the [CLS] token), all 768 dimensions
cls_embedding = last_hidden_state[0, 0, :]

first_five_sum = cls_embedding[:5].sum().item()
print("Sum of first 5 values in [CLS] embedding:", round(first_five_sum, 4))

Sum of first 5 values in [CLS] embedding: -1.2001


## Q8 — Attention weight from [CLS] to the word "fusion"

**What's happening:** Attention weights show how much each token "looks at" every other token when building its representation. Setting `output_attentions=True` makes the model return these weights. We then dig into: the *last* layer (`[-1]`), the *first* attention head (`[0]`), and specifically how much the `[CLS]` token attends to the token for the word "fusion."

Note: tokenizers sometimes split uncommon words into multiple sub-word pieces — we print the tokens first to confirm "fusion" appears as a single, whole token before locating its index.

In [11]:
# Reload the model, this time asking it to also return attention weights
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs2 = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs2 = model_attn(**inputs2)

# outputs2.attentions is a tuple of tensors, one per layer
# each tensor's shape is (batch, num_heads, seq_len, seq_len)
attentions = outputs2.attentions
last_layer_attention = attentions[-1]
first_head_attention = last_layer_attention[0, 0]  # batch 0, head 0

# Turn input_ids back into readable tokens so we can find "fusion"'s position
tokens = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])
print("Tokens:", tokens)

fusion_index = tokens.index("fusion")
print("Token index for 'fusion':", fusion_index)

# Row 0 = how much [CLS] attends to every other token; pick out the column for "fusion"
cls_to_fusion = first_head_attention[0, fusion_index].item()
print("Attention weight from [CLS] to 'fusion':", round(cls_to_fusion, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Token index for 'fusion': 4
Attention weight from [CLS] to 'fusion': 0.1025


## Q9 — Cosine similarity with Sentence-BERT (MiniLM)

**What's happening:** `all-MiniLM-L6-v2` is a smaller, faster model specifically designed to turn a whole sentence into *one* meaningful embedding vector (unlike raw BERT, which gives one vector per token). `.encode()` does this in one line. `cos_sim()` then measures how similar two vectors are — 1.0 means identical direction, 0 means unrelated.

In [12]:
from sentence_transformers import SentenceTransformer, util

sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

row0 = dataset[0]
prompt_embedding = sbert_model.encode(row0["prompt"])
option_b_embedding = sbert_model.encode(row0["B"])

similarity = util.cos_sim(prompt_embedding, option_b_embedding)
print("Cosine similarity (Prompt vs Option B), Row ID 0:", round(similarity.item(), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine similarity (Prompt vs Option B), Row ID 0: 0.7658


## Q10 — Comparing two full ranking pipelines: TF-IDF vs MiniLM

**What's happening:** We build two complete systems that each guess the top-3 most likely correct answers for every question, then compare them:
- **Pipeline 1 (TF-IDF):** the same word-overlap-based approach from Milestone 1.
- **Pipeline 2 (MiniLM):** uses sentence embeddings, which capture *meaning* rather than just matching words.

We measure MiniLM's overall MAP@3, and also count how many questions MiniLM "rescues" — cases where TF-IDF's top-3 missed the correct answer, but MiniLM's top-3 caught it. This shows concretely where meaning-based similarity beats simple word-overlap.

In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
options = ["A", "B", "C", "D", "E"]

# ---------- Pipeline 1: TF-IDF (same approach as Milestone 1) ----------
combined_texts = (
    train_df["prompt"] + " " + train_df["A"] + " " + train_df["B"] + " " +
    train_df["C"] + " " + train_df["D"] + " " + train_df["E"]
)

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_vectorizer.fit(combined_texts)

prompt_tfidf = tfidf_vectorizer.transform(train_df["prompt"])
option_tfidf = {opt: tfidf_vectorizer.transform(train_df[opt]) for opt in options}

def get_top3_tfidf(i):
    sims = {
        opt: sk_cosine_similarity(prompt_tfidf[i], option_tfidf[opt][i])[0][0]
        for opt in options
    }
    return sorted(sims, key=sims.get, reverse=True)[:3]

# ---------- Pipeline 2: MiniLM sentence embeddings ----------
# Encode every prompt and every option column once upfront (much faster than one-by-one)
prompt_embeddings = sbert_model.encode(train_df["prompt"].tolist())
option_embeddings = {opt: sbert_model.encode(train_df[opt].tolist()) for opt in options}

def get_top3_minilm(i):
    sims = {
        opt: util.cos_sim(prompt_embeddings[i], option_embeddings[opt][i]).item()
        for opt in options
    }
    return sorted(sims, key=sims.get, reverse=True)[:3]

# Same MAP@3 scoring function from Milestone 1
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    for idx, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (idx + 1)
    return 0.0

minilm_scores = []
rescued_count = 0  # TF-IDF missed it, but MiniLM caught it

for i in range(len(train_df)):
    actual = train_df.iloc[i]["answer"]

    tfidf_top3 = get_top3_tfidf(i)
    minilm_top3 = get_top3_minilm(i)

    minilm_scores.append(apk(actual, minilm_top3))

    if actual not in tfidf_top3 and actual in minilm_top3:
        rescued_count += 1

minilm_map3 = sum(minilm_scores) / len(minilm_scores)

print(f"MiniLM Pipeline MAP@3: {minilm_map3:.6f}")
print(f"Questions where TF-IDF missed but MiniLM caught the answer: {rescued_count}")

MiniLM Pipeline MAP@3: 0.423083
Questions where TF-IDF missed but MiniLM caught the answer: 502


## Q11 — Zero-shot classification (Softmax)

**What's happening:** Zero-shot classification lets a model pick the best label out of a set of *candidate labels* — even though it was never specifically trained on those exact labels. Here, we treat each MCQ option as a "candidate label" for the prompt (treated as the text to classify). By default, the model uses **softmax**, which forces all the label scores to sum to 1.0 — so higher confidence in one option directly lowers the others.

In [16]:
from transformers import pipeline

# Defaults to facebook/bart-large-mnli if no model is specified
zero_shot_classifier = pipeline("zero-shot-classification")

# Zero-indexing: "2nd row" = index 1
row1 = dataset[1]
candidate_labels = [row1["A"], row1["B"], row1["C"]]

result = zero_shot_classifier(row1["prompt"], candidate_labels)
print(result)

top_score = result["scores"][0]  # results are already sorted, highest first
print("Top-ranked option probability:", round(top_score, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

## Q12 — Zero-shot classification (independent Sigmoids)

**What's happening:** Setting `multi_label=True` changes the underlying math: instead of one softmax forcing scores to sum to 1, each label gets its own independent **sigmoid** score — as if asking "is this label true, yes or no?" separately for each one. That means the 3 scores are no longer forced to sum to 1, which is exactly what this question is testing.

In [17]:
result_multi = zero_shot_classifier(row1["prompt"], candidate_labels, multi_label=True)
print(result_multi)

sum_softmax = sum(result["scores"])       # from Q11 — forced to sum close to 1.0
sum_sigmoid = sum(result_multi["scores"])  # from this question — independent, no such constraint

difference = abs(sum_softmax - sum_sigmoid)
print("Sum (softmax, Q11):", sum_softmax)
print("Sum (independent sigmoids, this question):", sum_sigmoid)
print("Absolute difference:", round(difference, 4))

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

## Q13 — Generative AI: asking a small language model directly

**What's happening:** Instead of classifying, we now *generate* text. `flan-t5-small` is a small instruction-following model — you can literally ask it a question in plain English and it tries to answer in words, rather than picking from fixed labels. `max_new_tokens=5` limits how long its answer can be, since we only expect a single letter back.

In [27]:
!pip install -U "transformers==4.46.3" --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.7 MB/s eta 0:00:00:00:01


In [28]:
generator = pipeline("text2text-generation", model="google/flan-t5-small")

row0 = dataset[0]

prompt_text = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

output = generator(prompt_text, max_new_tokens=5)
print(output)
print("Model output:", output[0]["generated_text"])

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"